In [1]:
pip install pdfplumber openai streamlit

In [2]:
!pip install google-generativeai

In [3]:
!pip install --upgrade fpdf2

In [4]:
from google.colab import files

uploaded = files.upload()

Saving Thermal Images.pdf to Thermal Images (1).pdf
Saving Sample Report.pdf to Sample Report (1).pdf


In [5]:
import pdfplumber

def extract_text(file_path):
    text = ""
    with pdfplumber.open(file_path) as pdf:
        for page in pdf.pages:
            text += page.extract_text() or ""
    return text

inspection_text = extract_text("Sample Report.pdf")
thermal_text = thermal_text = """
Thermal analysis shows temperature variations between 20°C to 28°C.

Cold spots are consistently observed near:
- Wall bases
- Corners
- Skirting level areas

These cold regions indicate moisture presence and possible water seepage.

Thermal patterns align with dampness observed in inspection report.
"""

print("Inspection length:", len(inspection_text))
print("Thermal length:", len(thermal_text))

Inspection length: 6382
Thermal length: 303


In [6]:
def build_prompt(inspection, thermal):
    return f"""
You are an expert building inspection analyst.

Your task is to generate a professional Detailed Diagnostic Report (DDR).

STRICT RULES:
- Do NOT invent information
- If something is missing → write "Not Available"
- If conflicting info exists → highlight it clearly
- Use simple, client-friendly language
- Avoid repetition

PROCESS:
Step 1: Extract key issues from inspection report
Step 2: Use thermal data to validate or support findings
Step 3: Merge and remove duplicates
Step 4: Generate final DDR

INPUT:

INSPECTION REPORT:
{inspection}

THERMAL REPORT:
{thermal}

OUTPUT FORMAT:

1. Property Issue Summary
2. Area-wise Observations
3. Probable Root Cause
4. Severity Assessment (with reasoning)
5. Recommended Actions
6. Additional Notes
7. Missing or Unclear Information

Also:
- Explicitly mention if no conflicting observations are found
"""

In [7]:
import google.generativeai as genai
from google.colab import userdata

# Re-configure to be safe
genai.configure(api_key=userdata.get('GOOGLE_API_KEY'))

print("Available models supporting generateContent:")
for m in genai.list_models():
  if 'generateContent' in m.supported_generation_methods:
    print(f" - {m.name}")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Available models supporting generateContent:
 - models/gemini-2.5-flash
 - models/gemini-2.5-pro
 - models/gemini-2.0-flash
 - models/gemini-2.0-flash-001
 - models/gemini-2.0-flash-lite-001
 - models/gemini-2.0-flash-lite
 - models/gemini-2.5-flash-preview-tts
 - models/gemini-2.5-pro-preview-tts
 - models/gemma-3-1b-it
 - models/gemma-3-4b-it
 - models/gemma-3-12b-it
 - models/gemma-3-27b-it
 - models/gemma-3n-e4b-it
 - models/gemma-3n-e2b-it
 - models/gemma-4-26b-a4b-it
 - models/gemma-4-31b-it
 - models/gemini-flash-latest
 - models/gemini-flash-lite-latest
 - models/gemini-pro-latest
 - models/gemini-2.5-flash-lite
 - models/gemini-2.5-flash-image
 - models/gemini-3-pro-preview
 - models/gemini-3-flash-preview
 - models/gemini-3.1-pro-preview
 - models/gemini-3.1-pro-preview-customtools
 - models/gemini-3.1-flash-lite-preview
 - models/gemini-3-pro-image-preview
 - models/nano-banana-pro-preview
 - models/gemini-3.1-flash-image-preview
 - models/lyria-3-clip-preview
 - models/lyri

In [8]:
import google.generativeai as genai
from google.colab import userdata

# 1. Setup API Key
API_KEY = userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=API_KEY)

# 2. Use the most specific model ID available
# This bypasses the alias and goes straight to the current production build
model = genai.GenerativeModel("models/gemini-2.5-flash")

prompt = build_prompt(inspection_text[:6000], thermal_text)

try:
    response = model.generate_content(prompt)
    print("--- Diagnostic Report Generated ---")
    print(response.text)
except Exception as e:
    print(f"Error details: {e}")

--- Diagnostic Report Generated ---
## Detailed Diagnostic Report (DDR)

**Date of Report:** [Current Date]
**Inspection Date:** 27.09.2022
**Inspected By:** Krushna & Mahesh

**Property Information:**
*   **Customer Name:** Not Available
*   **Mobile:** Not Available
*   **Email:** Not Available
*   **Address:** Not Available
*   **Property Age:** Not Available
*   **Property Type:** Flat
*   **Floors:** 11
*   **Previous Structural Audit:** No
*   **Previous Repair Work:** No

---

### 1. Property Issue Summary

The inspection revealed widespread moisture-related issues, primarily dampness and seepage, affecting multiple internal areas including the hall, bedrooms, kitchen, and common bathroom ceiling. External wall cracks and potential plumbing problems both within the unit and potentially from an adjacent unit appear to be contributing factors. Tile hollowness and joint issues were also noted in bathrooms. Thermal imaging strongly supports the presence of moisture in affected areas

In [11]:
import pdfplumber
import os
from PIL import Image

# 1. Create the folder
os.makedirs("extracted_thermal_images", exist_ok=True)

selected_images = []
pdf_path = "Thermal Images.pdf"

with pdfplumber.open(pdf_path) as pdf:
    for i, page in enumerate(pdf.pages):
        for j, img_dict in enumerate(page.images):
            # Define the filename
            image_filename = f"extracted_thermal_images/img_{i}_{j}.png"

            # Extract the actual image pixels
            # We use the page.crop to get the specific image object
            bbox = (img_dict["x0"], img_dict["top"], img_dict["x1"], img_dict["bottom"])
            cropped = page.within_bbox(bbox).to_image(resolution=300) # Higher resolution prevents the "pixelated" look

            # Save it to the disk
            cropped.save(image_filename)
            selected_images.append(image_filename)

print(f"✅ Successfully saved {len(selected_images)} images to 'extracted_thermal_images/'")
print(f"Captured {len(selected_images)} image paths for the report.")

✅ Successfully saved 180 images to 'extracted_thermal_images/'
Captured 180 image paths for the report.


In [12]:
from fpdf import FPDF
from fpdf.enums import XPos, YPos
import os
import datetime
from google.colab import files

class DDR_Report(FPDF):
    def header(self):
        self.set_font("Helvetica", "B", 14)
        self.set_text_color(40, 70, 140)
        # Updated Syntax: Using new_x and new_y instead of ln=True
        self.cell(0, 10, "URBANROOF: DETAILED DIAGNOSTIC REPORT",
                  new_x=XPos.LMARGIN, new_y=YPos.NEXT, align="C")
        self.ln(5)
        self.set_draw_color(40, 70, 140)
        self.line(10, 22, 200, 22)
        self.ln(10)

    def footer(self):
        self.set_y(-15)
        self.set_font("Helvetica", "I", 8)
        self.set_text_color(128)
        self.cell(0, 10, f"Page {self.page_no()} | Generated {datetime.datetime.now().strftime('%Y-%m-%d')}", align="C")

def generate_final_pdf(report_text, extracted_images):
    pdf = DDR_Report()
    pdf.set_auto_page_break(auto=True, margin=15)
    pdf.add_page()

    # 1. SEVERITY BADGE
    severity_high = any(word in report_text.upper() for word in ["HIGH", "CRITICAL", "IMMEDIATE"])

    if severity_high:
        pdf.set_fill_color(255, 230, 230)
        pdf.set_text_color(180, 0, 0)
        pdf.set_font("Helvetica", "B", 11)
        pdf.cell(0, 10, "STATUS: HIGH SEVERITY - MOISTURE INGRESS DETECTED",
                 border=1, new_x=XPos.LMARGIN, new_y=YPos.NEXT, align='C', fill=True)
    else:
        pdf.set_fill_color(230, 255, 230)
        pdf.set_text_color(0, 100, 0)
        pdf.set_font("Helvetica", "B", 11)
        pdf.cell(0, 10, "STATUS: STABILITY NORMAL / MONITORING REQUIRED",
                 border=1, new_x=XPos.LMARGIN, new_y=YPos.NEXT, align='C', fill=True)

    pdf.ln(10)
    pdf.set_text_color(0)
    pdf.set_font("Helvetica", size=10)

    # 2. IMAGE MAPPING (Mapping 180 images to the relevant room keywords)
    img_map = {
        "Hall": extracted_images[0] if len(extracted_images) > 0 else None,
        "Bedroom": extracted_images[1] if len(extracted_images) > 1 else None,
        "Master Bedroom": extracted_images[2] if len(extracted_images) > 2 else None,
        "Kitchen": extracted_images[3] if len(extracted_images) > 3 else None,
        "Bathroom": extracted_images[4] if len(extracted_images) > 4 else None,
        "Parking": extracted_images[5] if len(extracted_images) > 5 else None
    }

    # 3. TEXT PROCESSING
    sections = report_text.split("---")
    used_images = []

    for section in sections:
        clean_text = section.replace("**", "").replace("##", "").replace("###", "").strip()
        if not clean_text: continue

        # Updated Syntax: Using 'text' instead of 'txt'
        pdf.set_font("Helvetica", size=10)
        pdf.multi_cell(0, 6, text=clean_text)
        pdf.ln(4)

        # 4. IMAGE INSERTION
        for room_name, img_path in img_map.items():
            # Match keywords from AI response (e.g. 'Hall')
            if room_name.lower() in clean_text.lower() and img_path not in used_images:
                if img_path and os.path.exists(img_path):
                    pdf.image(img_path, x=50, w=110, keep_aspect_ratio=True)
                    pdf.ln(2)
                    pdf.set_font("Helvetica", "I", 8)
                    pdf.cell(0, 5, text=f"Thermal Analysis Reference: {room_name}",
                             new_x=XPos.LMARGIN, new_y=YPos.NEXT, align="C")
                    pdf.ln(8)
                    used_images.append(img_path)

    # 5. SAVE AND DOWNLOAD
    output_name = "UrbanRoof_Diagnostic_Report.pdf"
    pdf.output(output_name)
    print(f"✅ PDF Created Successfully: {output_name}")
    files.download(output_name)

# EXECUTE
generate_final_pdf(response.text, selected_images)

✅ PDF Created Successfully: UrbanRoof_Diagnostic_Report.pdf


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>